## Join all files together

In [3]:
import pandas as pd
import numpy as np
import os
from datetime import timedelta

input_datadir = '/scratch/hydro4/users/kv25483/MetricEvaluation/Data/DanishRainData_Outputs'
resolutions = [5, 10, 30, 60]  # in minutes

all_events_list = []  # to store merged per-file data
total_events = 0

# Get the list of files (assuming all resolutions have the same filenames)
files = np.sort([f for f in os.listdir(os.path.join(input_datadir, "5mins_new_new")) if f.endswith(".csv")])

for num, file in enumerate(files[:31]):  # first 7 for testing
    # --- Step 1: read all resolutions ---
    dfs = {}
    for res in resolutions:
        df = pd.read_csv(os.path.join(input_datadir, f"{res}mins_new_new", file))
        df['event_num'] = df.index  # ensure we have event_num
        df['gauge_num'] = df['gauge_num'].apply(
            lambda x: int(x.split('_')[0]) if isinstance(x, str) and x.split('_')[0].isdigit() else x)
        
        # Add suffix to metrics to indicate resolution
        metric_cols = [c for c in df.columns if c not in ['event_num', 'gauge_num']]
        df.rename(columns={c: f"{c}_{res}m" for c in metric_cols}, inplace=True)
        dfs[res] = df

    # --- Step 2: find common events across all resolutions ---
    common_event_nums = set.intersection(*(set(df['event_num']) for df in dfs.values()))
    
    if not common_event_nums:
        print("No common event numbers")
        continue

    # Filter dfs to only include common events
    for res in resolutions:
        dfs[res] = dfs[res][dfs[res]['event_num'].isin(common_event_nums)].reset_index(drop=True)

    # --- Step 3: merge all resolutions side-by-side ---
    merged = dfs[5]
    for res in [10, 30, 60]:
        merged = merged.merge(dfs[res], on=['event_num', 'gauge_num'], how='inner')

    total_events += len(merged)
    print(f"{num} : {file}: {len(common_event_nums)} events retained across all resolutions, new total: {total_events}")

    all_events_list.append(merged)

# --- Step 4: concatenate all files ---
all_events_df = pd.concat(all_events_list).reset_index(drop=True)
print("Final shape:", all_events_df.shape)

0 : All_events_500520_precip_minute.csv: 1083 events retained across all resolutions, new total: 1083
1 : All_events_500920_precip_minute.csv: 944 events retained across all resolutions, new total: 2027
2 : All_events_5012_svk_precip_minute.csv: 293 events retained across all resolutions, new total: 2320
3 : All_events_501520_precip_minute.csv: 920 events retained across all resolutions, new total: 3240
4 : All_events_5025_svk_precip_minute.csv: 2667 events retained across all resolutions, new total: 5907
5 : All_events_5027_svk_precip_minute.csv: 2804 events retained across all resolutions, new total: 8711
6 : All_events_503120_precip_minute.csv: 1042 events retained across all resolutions, new total: 9753
7 : All_events_5032_svk_precip_minute.csv: 436 events retained across all resolutions, new total: 10189
8 : All_events_503520_precip_minute.csv: 1004 events retained across all resolutions, new total: 11193
9 : All_events_504020_precip_minute.csv: 237 events retained across all reso

### Get rid of small events

In [4]:
all_events_df = all_events_df[all_events_df['total_precip_5m']>4].copy()

### Get rid of big events

In [5]:
bad_indices = all_events_df.sort_values(by="total_precip_5m", ascending=False)[:15].index

# Drop rows at these indices from all dataframes
all_events_df = all_events_df.drop(index=bad_indices)

### Set I30 to 0 for 60m (from NAN) - and add _raw to the string

In [6]:
all_events_df["I30_60m"] = 0

In [7]:
all_events_df.columns = [
    col.replace('I30', 'I30_raw') if 'I30' in col else col
    for col in all_events_df.columns]

### Get rid of unwanted columns

In [8]:
# # Specify the suffixes to remove
# cols_to_remove = ['heaviest_half', 'T50']

# # Keep only columns that do NOT contain any of the above
# all_events_df = all_events_df[
#     [col for col in all_events_df.columns 
#      if not any(suffix in col for suffix in cols_to_remove)]].copy()

### Get version with just 5m data

In [9]:
# Specify the suffixes to remove
versions_to_remove = ['_dblnorm', '_dmc']

# Keep only columns that do NOT contain any of the above
events_raw = all_events_df[
    [col for col in all_events_df.columns 
     if not any(suffix in col for suffix in versions_to_remove)]].copy()
events_raw.columns = [col.replace('_raw', '') for col in events_raw.columns]

### Get version with just 5m data

In [10]:
# Specify the suffixes to remove
res_to_remove = ['_10m', '_30m', '_60m']

# Keep only columns that do NOT contain any of the above
events_5m = all_events_df[
    [col for col in all_events_df.columns 
     if not any(suffix in col for suffix in res_to_remove)]].copy()

events_5m.columns = [col.replace('_5m', '') for col in events_5m.columns]

### Save to file

In [11]:
output_datadir = '/scratch/hydro4/users/kv25483/MetricEvaluation/Data/'
all_events_df.to_csv(output_datadir + "IntermediateFiles/AllRes.csv",index=False)
events_5m.to_csv(output_datadir + "IntermediateFiles/Just5m.csv",index=False)
events_raw.to_csv(output_datadir + "IntermediateFiles/Justraw.csv",index=False)